Exercise: XGBoost

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

import sys
import os
sys.path.append(os.path.abspath('..')) # add parent directory to search path for modules

from src.utilities import read_filter_data

# Read data, filter rows with missing target, separate target from predictors
X_full, X_test_full, y = read_filter_data()

# Break off validation set from training data
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X_full, y, 
                                                                train_size=0.8, test_size=0.2,
                                                                random_state=0)

# Select categorical columns with relatively low cardinality (convenient but arbitrary)
low_cardinality_cols = [ # loops through columns, appends categorical columns that have < 10 unique values to list
    cname for cname in X_train_full.columns
    if X_train_full[cname].nunique() < 10
    and X_train_full[cname].dtype == "string"
]

# Select numerical columns
numerical_cols = [ # loops through columns, appends numerical columns to list
    cname for cname in X_train_full.columns
    if X_train_full[cname].dtype in ['int64', 'float64']
]

# Keep selected columns only
my_cols = low_cardinality_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid =  X_valid_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

# One-hot encode the data (to shorten the code, we use pandas)
X_train = pd.get_dummies(X_train) # converts categorical columns to numeric
X_valid = pd.get_dummies(X_valid)
X_test = pd.get_dummies(X_test)
X_train, X_valid = X_train.align(X_valid, join='left', axis=1) # align columns, using OneHotEncoder is still preferred over this, but this is just for practice
# join=left means keep all columns from X_train, modify X_valid to match X_train
X_train, X_test = X_train.align(X_test, join='left', axis=1)

Part A. build first model with gradient boosting

In [ ]:
from xgboost import XGBRegressor

# Define the model
my_model_1 = XGBRegressor(random_state=0)

# Fit the model
my_model_1.fit(X_train, y_train)

Part B. & C. fit, predict, and score model

In [ ]:
from src.utilities import score_model

mae_1 = score_model(my_model_1, X_train, X_valid, y_train, y_valid)
print(f"MAE for first model: {mae_1}")



Improving the model by:
- changing the defaul the default parameters in XGBRegressor
- then fit the model with training data
- predict and then use mean_absolute_error() to calculate MAE

Used chatGPT to recommend parameters and values to start with

In [ ]:
from sklearn.metrics import mean_absolute_error

# Define the model
my_model_2 = XGBRegressor(
    n_estimators=1000, # how many boosting cycles to run
    max_depth=5, # how deep each tree can be
    min_child_weight=3, # sets weight requirement for a leaf, high values make trees more conservative to splitting
    gamma=0, # set minimum loss reduction required to make a further split
    # high values allow splits with big loss reductions (improvement), lower values allow splits even for small improvements

    subsample=0.8, # 1.0 = use all rows (in data), < 1.0 = use subset of data
    # model could overfit if it sees all data, subsampling adds randomness which makes ensemble more robust

    colsample_bytree=0.8, # 1.0 = use all features (columns), < 1.0 = use subset of features
    # models could overfit if it sees all data, feature sampling adds randomnes, works in tandem with subsampling

    learning_rate=0.05, # how influential each cycle is
    # high values means the model makes big changes which can lead to overfitting
    # lower values means the model makes small changes which means it learns slower, needs more cycles (n_estimators), but usually has better generalization

    early_stopping_rounds=5, # control how many rounds of deteriorating scores allowed before stopping
    random_state=0, # seed
    n_jobs=-1 # how many cpu cores to use, -1 uses all available cores
)

# Fit the model
my_model_2.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
    )

# Evaluate and print score
mae_2 = mean_absolute_error(y_valid, my_model_2.predict(X_valid))
print(f"MAE for second model: {mae_2}")

Hyperparameter optimization with Optuna:
- instead of manually testing different parameter values and running the cell over and over, we can use the Optuna libary to automatically search for better model configurations.
- optuna works by trying different combinations of hyperparameters, evaluating their performance, and using the results to guide future trials toward better performing configurations (combinations of hyperparameters values).
- the cell below defines an objective function, which tells optuna what parameters to tune, how to train the model, and how it evaluate it's performance. optuna will then run multiple trials and attempt to find the set of parameters that minimizes the model's error.

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
import optuna

# Objective function
def objective(trial):
    """
    Objective function: given a set of hyperparameters (trial), train and evaluate model, then return MAE
    """
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
    }

    model = XGBRegressor(**param, random_state=0, n_jobs=-1) # **param unpacks the dictionary above into parameters

    # If data wasn't preprocessed already, making a pipeline before cross-validating would be neccessary

    score = -1 * cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error") # 5 folds

    return score.mean() # return average score

Below we run the Optuna study, which executes multiple trials of the objective function.
In each trial, Optuna selects a new set of hyperparameters, trains the model, and evaluates its performance. Over time, it uses previous results to influence its calculations toward better hyperparameter configurations

In [ ]:
# Create and run the optimization
study = optuna.create_study(direction="minimize") # direction=minimize as in our case, a lower MAE is better
study.optimize(objective, n_trials=30) # optuna calls objective function 30 times as it tries different hyperparameters

In [ ]:
# Get best parameter
best_param = study.best_params
print(",\n".join(f"{hyperparam}={value}" for hyperparam, value in best_param.items())) # print hyperparameters in copy and paste format
# best_param.items() returns a list of tuples (hyperparameter, value)
# ",\n" joins the tuples with a comma and a newline

# Get best score
best_score = study.best_value
print(f"\nBest Score: {best_score}")

The best parameters will slightly vary each time the optimization is run. For this run, the best parameters are:

n_estimators=606,
max_depth=5,
min_child_weight=1,
gamma=1.5147914838456218,
subsample=0.5501061768440515,
colsample_bytree=0.6413071011659919,
learning_rate=0.03792611850810067

In [ ]:
# Define the model
my_model_best = XGBRegressor(
    n_estimators=606,
    max_depth=5,
    min_child_weight=1,
    gamma=1.5147914838456218,
    subsample=0.5501061768440515,
    colsample_bytree=0.6413071011659919,
    learning_rate=0.03792611850810067,
    
    early_stopping_rounds=5, # optuna tuned n_estimators already, using early stopping ensures training stops early if validation deteriorates
    random_state=0,
    n_jobs=-1
)

# or simply:
# my_model_best = XGBRegressor(**best_param, random_state=0, n_jobs=-1)

# Fit the model
my_model_best.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

# Evaluate and print score
mae_best = mean_absolute_error(y_valid, my_model_best.predict(X_valid))
print(f"MAE for model with optimized hyperparameters: {mae_best}")

MAE for first model: 18161.826171875

MAE for model with chatGPT recommended parameters: 16586.744140625

MAE for model with optimized hyperparameters: 16475.44921875

Improvements were incremental at best, and important thing to remember is that hyperparameter tuning won't save your model.
The biggest improvements come from quality of data (better features = better performance), feature engineering (manipulating data/features), preprocessing decisions (e.g. handling missing values, encoding categories, etc), and model choice as we saw a big improvement going from random forests to XGBoost regressors.